# Artificial and Computational Intelligence Assignment 1 (Group 153)

## Problem solving by Uninformed & Informed Search

| SL.NO | BITS ID      | NAME                      | CONTRIBUTION |
|-------|-------------|---------------------------|--------------|
| 1     | 2024AA05731 | Suraj Anand               | 20%          |
| 2     | 2024AA05732 | Ashvini Satish Bhosale    | 20%          |
| 3     | 2024AA05733 | Bollam Manindra           | 20%          |
| 4     | 2024AA05734 | S. Hari Chandana          | 20%          |
| 5     | 2024AA05735 | Nikhil Yadav              | 20%          |

Things to follow
1.	Use appropriate data structures to represent the graph and the path using python libraries
2.	Provide proper documentation
3.	Find the path and print it

### 1.	Define the environment in the following block

List the PEAS decription of the problem here in this markdown block

Design the agent as PSA Agent(Problem Solving Agent)
Clear Initial data structures to define the graph and variable declarations is expected
IMPORTATANT: Write distinct code block as below

In [85]:
#Code Block : Set Initial State (Must handle dynamic inputs)
import random
import time

def set_initial_state():
    # Initialize and return a list of initial locations
    print('user input is case sensitive:')
    locations = ['Admission Office', 'Hostel Office', 'Library', 'Canteen', 'Department visit', 'Hostel visit']
    return locations

In [86]:
#Code Block : Set the matrix for transition & cost (as relevant for the given problem)
def set_transition_cost_matrix(): 
    graph = {
        'Admission Office': {'Hostel Office': 2, 'Library': 4},
        'Hostel Office': {'Admission Office': 2, 'Library': 4, 'Canteen': 6, 'Hostel visit': 2},
        'Library': {'Admission Office': 4, 'Canteen': 7, 'Hostel Office': 4, 'Department visit': 3},
        'Canteen': {'Hostel Office': 6, 'Library': 7, 'Department visit': 2, 'Hostel visit': 6, 'Exit': 8},
        'Department visit': {'Library': 3, 'Canteen': 2, 'Exit': 5},
        'Hostel visit': {'Hostel Office': 2, 'Canteen': 6, 'Exit': 4},
        'Exit': {}
    }
    return graph

In [87]:
#Code Block : Write function to design the Transition Model/Successor function. Ideally this would be called while search algorithms are implemented

def get_successors(graph, node):
    return graph[node]

In [88]:
#Code block : Write function to handle goal test (Must handle dynamic inputs). Ideally this would be called while search algorithms are implemented

def goal_test(path, goal, locations):
    # Check if the path includes all locations and ends at the goal
    return len(path) == len(locations) and set(path) == locations and path[-1] == goal

### 2.	Definition of Algorithm 1 (Greedy Best First Search)

In [89]:
#Code Block : Function for algorithm 1 implementation

"""
    h(n) = (alpha * MST cost of unvisited nodes) + (beta * Direct cost to exit)
    The Prim's algorithm is used to compute the MST cost for all unvisited nodes
    alpha: Weight for MST cost component (default = 1).
    beta: Weight for direct cost to exit component (default = 2).
    
"""
import heapq

def prim_mst_cost(graph, unvisited):
    """
    Computes the Minimum Spanning Tree (MST) cost for unvisited nodes using Prim's algorithm.
    """
    if not unvisited:
        return 0  # If no unvisited nodes, MST cost is zero.

    start_node = next(iter(unvisited))  # Select an arbitrary node as the starting point.
    min_heap = [(0, start_node)]  # Min-heap storing (edge weight, node).
    visited = set()
    mst_cost = 0  # Initialize MST cost.

    while min_heap:
        cost, node = heapq.heappop(min_heap)  # Extract node with the minimum edge cost.
        if node in visited:
            continue  # Skip if the node is already visited.
        
        visited.add(node)
        mst_cost += cost  # Add the edge weight to the MST cost.

        # Explore neighbors and add their edges to the heap if not visited.
        for neighbor, weight in graph[node].items():
            if neighbor in unvisited and neighbor not in visited:
                heapq.heappush(min_heap, (weight, neighbor))

    return mst_cost  # Return the total MST cost.


# alpha = 1  # Weight for MST
# beta = 2   # Weight for direct cost
def compute_heuristics(graph, nodes, exit_node, alpha=1, beta=2):
    """
    Computes heuristics for all nodes using:
    - MST cost for unvisited nodes (alpha weight)
    - Direct cost from the current node to the exit node (beta weight)
    """
    heuristics = {}
    for node in nodes:
        if node == exit_node:
            heuristics[node] = 0  # Heuristic for the exit node is always 0
        else:
            # Remaining unvisited nodes excluding the current node and exit node
            unvisited = set(nodes) - {node, exit_node}
            
            # Compute MST cost for the unvisited nodes
            mst_cost = prim_mst_cost(graph, unvisited)
            
            # Compute the direct cost from the current node to the exit (if exists)
            direct_exit_cost = graph[node].get(exit_node, float('inf'))

            # Calculate weighted direct cost if a direct path exists
            if direct_exit_cost != float('inf'):
                weighted_direct_cost = beta * direct_exit_cost
            else:
                weighted_direct_cost = 0

            # Combine MST cost and weighted direct cost to form the heuristic
            heuristics[node] = alpha * mst_cost + weighted_direct_cost
    
    return heuristics


def greedy_best_first_search(graph, start, goal):
    """
    Perform Greedy Best-First Search on the given graph.
    
    Parameters:
    graph (dict): The graph representation with nodes and edges.
    start (str): The starting node.
    goal (str): The goal node.
    
    Returns:
    list: The path from start to goal if found, otherwise None.
    """
    
    # Compute heuristics for all nodes
    heuristic = compute_heuristics(graph, set(graph.keys()), goal)
    print('Heuristics', heuristic)
    
    # Initialize the priority queue with the start node
    priority_queue = [(heuristic[start], [start])]
    
    max_queue_size = 0  # To track space usage
    total_edges_explored = 0  # To track the number of edges explored
 
    def get_priority(item):
        return item[0]

    while priority_queue:
        # Sort the priority queue based on heuristic values
        priority_queue.sort(key=get_priority)  # O(N log N)
        
        # Pop the node with the lowest heuristic value
        _, path = priority_queue.pop(0)  # O(N)
        current_node = path[-1]
        
        # Check if the goal has been reached
        if goal_test(path, goal, set(graph.keys())):
            print(f"Time Complexity: O({len(graph)} × {total_edges_explored} log {len(graph)}) ≈ O(N × E log N)")
            print(f"Space Complexity: O({len(graph)} + {max_queue_size}) ≈ O(N + E)\n")
            return path
        
        # Explore the neighbors of the current node
        for neighbor in get_successors(graph, current_node):
            total_edges_explored += 1
            if neighbor not in path or neighbor == goal:
                new_path = list(path)
                new_path.append(neighbor)
                priority_queue.append((heuristic[neighbor], new_path))
        
        # Update the maximum size of the priority queue
        max_queue_size = max(max_queue_size, len(priority_queue))
    
    # If the goal is not reached, return None
    print(f"Time Complexity: O({len(graph)} × {total_edges_explored} log {len(graph)}) ≈ O(N × E log N)")
    print(f"Space Complexity: O({len(graph)} + {max_queue_size}) ≈ O(N + E)\n")
    return None

### 3.	Definition of Algorithm 2 Genetic Algorithm

In [90]:
#Code Block : Function for algorithm 2 implementation

def fitness(graph, path):
    """
    Calculate the total distance of the given path in the graph.
    If the path is invalid, return infinity.
    """
    total_distance = 0
    for i in range(len(path) - 1):
        try:
            total_distance += graph[path[i]][path[i + 1]]
        except KeyError:
            return float('inf')
    return total_distance

def generate_population(size, start):
    """
    Generate an initial population of individuals.
    Each individual is a path starting from the start node and ending at 'Exit'.
    """
    population = []
    locations = list(graph.keys())
    locations.remove(start)
    locations.remove('Exit')
    for _ in range(size):
        individual = [start] + random.sample(locations, len(locations)) + ['Exit']
        population.append(individual)
    return population

def get_fitness(individual):
    """
    Get the fitness of an individual.
    Fitness is defined as the total distance of the path.
    """
    return fitness(graph, individual)

def selection(population):
    """
    Select the top 50% of the population based on fitness.
    """
    population.sort(key=get_fitness)
    return population[:len(population) // 2]

def crossover(parent1, parent2):
    """
    Perform crossover between two parents to produce a child.
    The child inherits the first half from parent1 and the rest from parent2.
    """
    child = parent1[:len(parent1) // 2]
    for gene in parent2:
        if gene not in child:
            child.append(gene)
    return child

def mutate(individual):
    """
    Mutate an individual by swapping two random genes (excluding start and end).
    """
    idx1 = random.randint(1, len(individual) - 2)
    idx2 = random.randint(1, len(individual) - 2)
    individual[idx1], individual[idx2] = individual[idx2], individual[idx1]

def genetic_algorithm(graph, start, goal, population_size=100, generations=500):
    """
    Perform the Genetic Algorithm to find the optimal path from start to goal.
    """
    population = generate_population(population_size, start)
    
    # Time and Space Complexity Analysis
    chromosome_length = len(graph)
    print(f"Time Complexity : O({population_size} * {generations} * ({chromosome_length} + F)) ≈ O(G * P * (N + F)) ")
    print(f"Space Complexity : O({population_size} * {chromosome_length}) ≈ O(P * N)")
    
    for _ in range(generations):
        selected_population = selection(population)
        new_population = []
        
        while len(new_population) < population_size:
            parent1 = random.choice(selected_population)
            parent2 = random.choice(selected_population)
            child = crossover(parent1, parent2)
            if random.random() < 0.1:  # Mutation probability
                mutate(child)
            new_population.append(child)
        
        population = new_population
    
    best_individual = min(population, key=get_fitness)
    return best_individual

### DYNAMIC INPUT

IMPORTANT : Dynamic Input must be got in this section. Display the possible states to choose from:
This is applicable for all the relevent problems as mentioned in the question.

In [91]:
#Code Block : Function & call to get inputs (start/end state)
def get_inputs():
    # Get the initial set of locations
    locations = set_initial_state()
    
    # Display the possible states to choose from
    print("Possible states to choose from:")
    print(", ".join(locations))
    print()
    
    # Prompt the user to enter the starting point
    start_point = input("Enter the starting point: ")
    
    if(start_point == 'Exit'):
        return 'Exit'
    
    # Validate the starting point
    if start_point not in locations:
        return None
    
    return start_point

### 4.	Calling the search algorithms
(For bidirectional search in below sections first part can be used as per Hint provided. Under second section other combinations as per Hint or your choice of 2 algorithms can be called .As an analyst suggest suitable approximation in the comparitive analysis section)

In [92]:
#Invoke algorithm 1 (Should Print the solution, path, cost etc., (As mentioned in the problem))
def invoke_greedy_best_first_search(graph, start_point):
    if start_point is None:
        return

    start_time = time.time()  # Record start time
    optimal_path = greedy_best_first_search(graph, start_point, 'Exit')  # Find optimal path
    end_time = time.time()  # Record end time

    if optimal_path:
        print(f"The optimal path from {start_point} to Exit: {' -> '.join(optimal_path)}")
        print(f"Total cost: {fitness(graph, optimal_path)}")
    else:
        print(f"No path found from {start_point} to Exit.")

    print(f"Time taken: {end_time - start_time} seconds")

In [93]:
#Invoke algorithm 2 (Should Print the solution, path, cost etc., (As mentioned in the problem))
def invoke_genetic_algorithm(graph,start_point):
    
    if start_point is None:
        return
    
    start_time = time.time()
    
    optimal_path = genetic_algorithm(graph, start_point, 'Exit')
    
    end_time = time.time()
    
    print(f"The optimal path from {start_point} to Exit is: {' -> '.join(optimal_path)}")
    print(f"Total cost: {fitness(graph, optimal_path)}")
    
    print(f"Time taken: {end_time - start_time} seconds")

In [94]:
#Main function to run the algorithms and compare them
if __name__ == "__main__":
    graph = set_transition_cost_matrix()
    
    start_point = get_inputs()
    
    if(start_point==None):
        print("Invalid starting point. Exiting the program.")
    elif(start_point=='Exit'):
        print("Starting point cannot be the Exit. Exiting the program.")
    else: 
        print("Running Greedy Best First Search Algorithm:")
        invoke_greedy_best_first_search(graph,start_point)

        print("\nRunning Genetic Algorithm:")
        invoke_genetic_algorithm(graph,start_point)

user input is case sensitive:
Possible states to choose from:
Admission Office, Hostel Office, Library, Canteen, Department visit, Hostel visit

Running Greedy Best First Search Algorithm:
Heuristics {'Canteen': 27, 'Department visit': 24, 'Hostel Office': 15, 'Hostel visit': 19, 'Library': 12, 'Admission Office': 11, 'Exit': 0}
Time Complexity: O(7 × 115 log 7) ≈ O(N × E log N)
Space Complexity: O(7 + 11) ≈ O(N + E)

The optimal path from Admission Office to Exit: Admission Office -> Hostel Office -> Hostel visit -> Canteen -> Library -> Department visit -> Exit
Total cost: 25
Time taken: 0.0010020732879638672 seconds

Running Genetic Algorithm:
Time Complexity : O(100 * 500 * (7 + F)) ≈ O(G * P * (N + F)) 
Space Complexity : O(100 * 7) ≈ O(P * N)
The optimal path from Admission Office to Exit is: Admission Office -> Library -> Hostel Office -> Hostel visit -> Canteen -> Department visit -> Exit
Total cost: 23
Time taken: 0.15604853630065918 seconds


### 5.	Comparitive Analysis (Time and Space Complexity)

In [95]:
#Code Block : Print the Time & Space complexity of Greedy Best First Search
print("\nTime Complexity of Greedy Best First Search: O(N × E log N)")
print("Space Complexity of Greedy Best First Search: O(N + E)")


Time Complexity of Greedy Best First Search: O(N × E log N)
Space Complexity of Greedy Best First Search: O(N + E)


In [96]:
#Code Block : Print the Time & Space complexity of Genetic Algorithm
print("\nTime Complexity of Genetic Algorithm: O(G * P * (N + F))")
print("Space Complexity of Genetic Algorithm: O(P * N)")


Time Complexity of Genetic Algorithm: O(G * P * (N + F))
Space Complexity of Genetic Algorithm: O(P * N)


### 6.	Provide your comparitive analysis or findings in no more than 3 lines in below section

Comparison : 
GBFS is faster but can get stuck in local optima, while GA avoids local optima but is slower due to evolutionary iterations. Using Prim's Algorithm in GBFS increases heuristic accuracy but adds computational load O(N x E LogN) whereas hop based heuristics reduces time complexity to O(E Log N).GA is more suitable for complex search spaces where an optimal solution is not easily found using heuristics alone, making it ideal for problems requiring exploration of multiple solutions.